# 01 — Classificatore a feature (TF-IDF + regressione logistica)

Sul dataset finale (`1-data-cleaning/03_dataset_finale`): solo il **testo degli articoli**.

Fold **per gruppo** (la valutazione principale: tutte le testate di un gruppo, cioè rete o stesso owner, nello
stesso fold) e **per testata** (confronto: i gruppi possono essere spezzati tra training e test). Un modello per fold: vocabolario e IDF si
stimano **solo sul training** del fold, il test passa solo da `transform`. Iperparametri fissi, nessuna scelta
guarda il test.

Per avere un metro, accanto al classificatore due baseline senza testo: la **classe di maggioranza** del
training e una **scelta a caso** con le proporzioni del training.

Salva `Results/2-classificazione/feature_<schema>_articoli.csv` (probabilità `p_l1..p_l5`, `y_pred`, `y_true`,
`domain`, `fold`) e le letture in `feature_metriche.csv`. Se un file c'è già, lo ricarica.

**Sul server:**

```bash
cd ~/John/WebScience2027/2-classificazione
screen -dmS feature papermill -k mbvenv 01_classificatore_feature.ipynb ../Results/2-classificazione/esecuzioni/01_classificatore_feature.ipynb
```

In [1]:
import time

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from comune import CLASSI, NOME_SCHEMA, SCHEMI, SEED, carica, cartella_risultati, letture, registratore

In [2]:
SCHEMI_DA_FARE = SCHEMI
N_JOBS = 5                 # fold in parallelo
PROVA = False              # True: 40 testate, risultati in Results/2-classificazione/prova/

In [3]:
OUT = cartella_risultati(PROVA)
registra = registratore(OUT / "avanzamento_feature.log")
art = carica(PROVA)
registra(f"=== dataset finale: {len(art)} articoli, {art.domain.nunique()} testate ===")

2026-10-16 15:05:52  === dataset finale: 29129 articoli, 193 testate ===


## Addestramento

Per ogni fold: `fit_transform` sul training, `transform` sul test.

In [4]:
def un_fold(schema, fold):
    t0 = time.time()
    tr, te = art[art[schema] != fold], art[art[schema] == fold]
    v = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True,
                        max_features=100_000, strip_accents="unicode")
    clf = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    clf.fit(v.fit_transform(tr.text), tr.label)              # vocabolario, IDF e modello: solo training
    p = clf.predict_proba(v.transform(te.text))              # test: solo transform
    registra(f"{schema} fold {fold}: fatto in {time.time() - t0:.0f} s ({len(tr)} training, {len(te)} test), "
             f"acc articoli {accuracy_score(te.label, clf.classes_[p.argmax(1)]):.3f}")
    return te.index.to_numpy(), clf.classes_, p


def come_pred(proba, schema):
    pred = pd.DataFrame(proba, columns=CLASSI)
    pred.insert(0, "domain", art.domain.values)
    pred.insert(0, "article_id", art.article_id.values)
    pred["fold"] = art[schema].values
    pred["y_true"] = art.label.values
    pred["y_pred"] = proba.argmax(1)
    return pred


def addestra(schema):
    file_out = OUT / f"feature_{schema}_articoli.csv"
    if file_out.exists():
        registra(f"{schema}: già fatto, carico {file_out.name}")
        return pd.read_csv(file_out)
    proba = np.zeros((len(art), 5))
    for idx, classi, p in Parallel(n_jobs=N_JOBS)(delayed(un_fold)(schema, f) for f in sorted(art[schema].unique())):
        proba[idx[:, None], classi[None, :]] = p
    pred = come_pred(proba, schema)
    pred.to_csv(file_out, index=False)
    return pred


pred = {s: addestra(s) for s in SCHEMI_DA_FARE}

2026-10-16 15:07:35  fold_gruppo fold 1: fatto in 101 s (23649 training, 5480 test), acc articoli 0.260


2026-10-16 15:07:37  fold_gruppo fold 3: fatto in 103 s (23563 training, 5566 test), acc articoli 0.267


2026-10-16 15:07:38  fold_gruppo fold 4: fatto in 103 s (23063 training, 6066 test), acc articoli 0.345


2026-10-16 15:07:44  fold_gruppo fold 5: fatto in 108 s (22483 training, 6646 test), acc articoli 0.316


2026-10-16 15:07:47  fold_gruppo fold 2: fatto in 113 s (23758 training, 5371 test), acc articoli 0.248


2026-10-16 15:09:30  fold_testata fold 1: fatto in 101 s (23163 training, 5966 test), acc articoli 0.369


2026-10-16 15:09:30  fold_testata fold 5: fatto in 99 s (23377 training, 5752 test), acc articoli 0.381


2026-10-16 15:09:33  fold_testata fold 4: fatto in 102 s (23127 training, 6002 test), acc articoli 0.450


2026-10-16 15:09:35  fold_testata fold 2: fatto in 106 s (23120 training, 6009 test), acc articoli 0.356


2026-10-16 15:09:39  fold_testata fold 3: fatto in 109 s (23729 training, 5400 test), acc articoli 0.461


## Baseline senza testo

Per ogni fold, dalle testate di training: la classe più frequente (per tutti gli articoli del test), e una
classe estratta a caso con le proporzioni del training (seed fisso).

In [5]:
def baseline(schema, modo):
    rng = np.random.default_rng(SEED)
    y = np.zeros(len(art), dtype=int)
    for fold in sorted(art[schema].unique()):
        tr, te = art[art[schema] != fold], art[art[schema] == fold]
        priori = tr.drop_duplicates("domain").label.value_counts(normalize=True).sort_index()
        if modo == "maggioranza":
            y[te.index] = priori.idxmax()
        else:                                  # a caso, una classe per testata (il voto non deve "mediare" il caso)
            per_testata = dict(zip(te.domain.unique(), rng.choice(priori.index, te.domain.nunique(), p=priori.values)))
            y[te.index] = te.domain.map(per_testata)
    return pd.DataFrame({"domain": art.domain, "y_true": art.label, "y_pred": y})

## Risultati

Per articolo e per testata (voto di maggioranza): accuratezza a 5 classi, F1 macro, binario (soglia tra
$l_2$ e $l_3$, punteggio NewsGuard 60) e tolerant ($\alpha$ = 0.5 e 1).

In [6]:
righe = []
for s in SCHEMI_DA_FARE:
    righe.append({"modello": "TF-IDF + LR", "fold": NOME_SCHEMA[s], **letture(pred[s])})
    for modo in ["maggioranza", "a caso"]:
        righe.append({"modello": f"baseline: {modo}", "fold": NOME_SCHEMA[s], **letture(baseline(s, modo))})
ris = pd.DataFrame(righe).set_index(["fold", "modello"])
ris.to_csv(OUT / "feature_metriche.csv")
ris[[c for c in ris.columns if c.startswith("testata")] + ["articolo 5cl", "articolo bin"]]

testata 5cl  testata F1  testata bin  \
fold        modello                                                       
per gruppo  TF-IDF + LR                  0.321       0.274        0.653   
            baseline: maggioranza        0.321       0.097        0.606   
            baseline: a caso             0.244       0.199        0.528   
per testata TF-IDF + LR                  0.435       0.401        0.741   
            baseline: maggioranza        0.321       0.097        0.606   
            baseline: a caso             0.233       0.202        0.560   

                                   testata tol0.5  testata tol1  articolo 5cl  \
fold        modello                                                             
per gruppo  TF-IDF + LR                     0.526         0.731         0.289   
            baseline: maggioranza           0.464         0.606         0.309   
            baseline: a caso                0.389         0.534         0.245   
per testata TF-IDF + LR                     0.611         0.788         0.402   
            baseline: maggioranza           0.464         0.606         0.309   
            baseline: a caso                0.409         0.585         0.242   

                                   articolo bin  
fold        modello                              
per gruppo  TF-IDF + LR                   0.641  
            baseline: maggioranza         0.597  
            baseline: a caso              0.526  
per testata TF-IDF + LR                   0.721  
            baseline: maggioranza         0.597  
            baseline: a caso              0.536

In [7]:
# per testata, fold per gruppo: matrice di confusione e accuratezza per livello
from comune import per_testata
t = per_testata(pred["fold_gruppo"])
LIV = ["l1", "l2", "l3", "l4", "l5"]
display(pd.crosstab(t.y_true.map(dict(enumerate(LIV))), t.y_pred.map(dict(enumerate(LIV))), rownames=["vero"], colnames=["predetto"]))
(t.y_true == t.y_pred).groupby(t.y_true.map(dict(enumerate(LIV)))).agg(["mean", "size"]).round(2).T

predetto,l1,l2,l3,l4,l5
vero,,,,,
l1,7,7,6,7,0
l2,7,23,6,9,4
l3,1,13,3,8,3
l4,2,14,3,26,17
l5,0,5,1,18,3


y_true,l1,l2,l3,l4,l5
mean,0.26,0.47,0.11,0.42,0.11
size,27.00,49.00,28.00,62.00,27.00
